# RepairScope-X — Combined NB21 Completion + NB29 Semantic-Failure Repair

**One notebook, two isolated real-GPU phases, one research package.**

- **Phase A (MDLM, Transformers 4.57.0):** restore the genuine 48 saved generation results if needed, select the original frozen eight failures, finish the missing **64 native MDLM infill repairs** under the exact NB21 protocol, and export its separate results.
- **Phase B (ALoDLM, Transformers 4.56.1):** after **Runtime → Restart session** (mandatory; do not run both model stacks in the same kernel), collect **genuine, identifiable wrong semantic answers**, freeze disjoint calibration and held-out cohorts, test fixed/global/adaptive policies and compute cost, then export.
- **Never mix the two trial populations.** NB21 is a replication/closure study of earlier data; NB29 is a newly defined prospective study.

*Execution order:* Run Phase A cells until the NB21 completion gate and Drive backup. Then restart the runtime, **skip all Phase A cells**, and run Phase B from its installation cell. Google Drive saves each trial immediately. Every batch is restart-safe. Colab free-T4 availability and total time cannot be guaranteed; the stages may span multiple sessions.

**Scientific gate:** NB29 eligibility requires an inspectable **incorrect** final semantic answer, not merely missing `FINAL_ANSWER`, a correct `\boxed{n}`, or truncated reasoning. If quota is not reached, the notebook stops rather than silently weakening that requirement.


## PHASE A — Notebook 21 MDLM protocol (original working cells preserved)

**Required environment:** fresh T4 Colab. The original 48-generation records are embedded below solely for checkpoint restoration; new 128/256-token generation is skipped once saved. Real GPU smoke inference is required.

### 0. Safe Python library setup — first code cell

In [ ]:
# Must be first code cell; does not import NumPy into the notebook kernel.
import os, sys, subprocess, pathlib
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'  # public model snapshots
if sys.version_info < (3,10):
    raise RuntimeError('Python 3.10 or newer is required.')
check="import numpy, torch; assert numpy.asarray([1,2]).sum()==3; print('NumPy',numpy.__version__,'Torch',torch.__version__)"
r=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-7000:])
    raise RuntimeError('Scientific Python environment already corrupted. Colab: Runtime > Disconnect and delete runtime; reconnect; do NOT reinstall NumPy inside a live runtime.')
print(r.stdout)
# Narrow, pinned Python interfaces only; never reinstall PyTorch, NumPy, SciPy or Pandas.
packages=['transformers==4.57.0','accelerate==1.11.0',
          'huggingface-hub==0.35.3','tokenizers==0.22.1','safetensors==0.6.2']
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','--no-deps',*packages],check=True)
preflight="import torch,transformers,accelerate,huggingface_hub,tokenizers,safetensors; from transformers import AutoTokenizer; from transformers.processing_utils import Unpack; from transformers.utils import TransformersKwargs; from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask; print('model interface',transformers.__version__)"
r=subprocess.run([sys.executable,'-c',preflight],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-9000:])
    raise RuntimeError('Model library import preflight failed. Start from a fresh runtime and report this error rather than running GPU stages.')
print(r.stdout)

### 1. Colab runtime + exact source revision + persistent checkpoint folder

In [ ]:
import base64, zlib, json, re, math, ast as pyast, gc, time, random, traceback, hashlib, zipfile, csv, shutil, subprocess, sys, importlib.util, os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'Choose T4 GPU and restart from the beginning.'
assert transformers.__version__=='4.57.0', 'Transformers version mismatch; restart with a fresh runtime.'
GPU=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
assert VRAM >= 7, f'Insufficient GPU memory: {VRAM:.2f} GB'
print('GPU',GPU,'VRAM',round(VRAM,2),'GB','torch',torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/RepairScopeX/RSX21_MDLMCostControlled_v1')
DRIVE.mkdir(parents=True,exist_ok=True)
# Verify persistent destination is writable; do not fall back to ephemeral /content.
(DRIVE/'drive_write_test.txt').write_text('ok',encoding='utf-8')
assert (DRIVE/'drive_write_test.txt').read_text()=='ok'
SOURCE_COMMIT='ca176752fbceec49c6b4777a2c18ae88e4eb10ed'
ROOT=Path('/content/rsx_v6_official_repo')
if not (ROOT/'.git').exists():
    ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','init','-q',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'remote','add','origin','https://github.com/ZHZisZZ/dllm.git'],check=True)
try:
    sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
except subprocess.CalledProcessError:
    sha=''
if sha!=SOURCE_COMMIT:
    for retry in range(3):
        try:
            subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','--depth','1','origin',SOURCE_COMMIT],check=True,timeout=240)
            subprocess.run(['git','-C',str(ROOT),'checkout','-q','--force','FETCH_HEAD'],check=True,timeout=80)
            break
        except Exception:
            if retry==2: raise
            time.sleep(2**retry)
sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert sha==SOURCE_COMMIT
print('Official code SHA',sha)

### 2. Load byte-identical official sampler source with inference-only package initializers

In [ ]:
# Exact official sampler files, copied BYTE-FOR-BYTE from the pinned upstream commit.
SHIM=Path('/content/rsx_v6_inference_only')
files=[
    'dllm/core/samplers/__init__.py','dllm/core/samplers/base.py',
    'dllm/core/samplers/mdlm.py','dllm/core/samplers/bd3lm.py',
    'dllm/core/samplers/utils.py','dllm/core/schedulers/__init__.py',
    'dllm/core/schedulers/alpha.py','dllm/core/schedulers/kappa.py',
]
for rel in files:
    source=ROOT/rel; destination=SHIM/rel
    assert source.exists(),source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert hashlib.sha256(source.read_bytes()).digest()==hashlib.sha256(destination.read_bytes()).digest()
for rel in ['dllm/__init__.py','dllm/core/__init__.py']:
    f=SHIM/rel;f.parent.mkdir(parents=True,exist_ok=True)
    f.write_text('# Inference namespace, without unrelated training imports.\n')
# Model class from exact official source, no checkpoint `trust_remote_code` needed.
MODEL_FILE=ROOT/'dllm/pipelines/a2d/models/qwen2/modeling_qwen2.py'
assert MODEL_FILE.is_file()
sys.path.insert(0,str(SHIM))
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig, BD3LMSampler, BD3LMSamplerConfig
model_module_name='rsx_v6_a2d_qwen2_source'
if model_module_name not in sys.modules:
    spec=importlib.util.spec_from_file_location(model_module_name,str(MODEL_FILE))
    module=importlib.util.module_from_spec(spec)
    sys.modules[model_module_name]=module
    spec.loader.exec_module(module)
else:
    module=sys.modules[model_module_name]
A2DQwen2Config=module.A2DQwen2Config
A2DQwen2LMHeadModel=module.A2DQwen2LMHeadModel
assert A2DQwen2Config.model_type=='a2d-qwen2'
print('Official samplers:',MDLMSampler.__name__,BD3LMSampler.__name__)

### 3. Frozen 24-problem benchmark (identical to NB20)

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

### 4. Task-aware answer verifier and regression tests

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

### 5. The **actual** earlier Notebook-20 MDLM reference records (64 total-step settings)

In [ ]:
REFERENCE_SHA256 = "aa06e0a452dffb1b3d8755c132acd9188b1faa3d9ab53548d996478cd795a7f1"
_ref_bytes=zlib.decompress(base64.b64decode("""..."""))
assert hashlib.sha256(_ref_bytes).hexdigest()==REFERENCE_SHA256
UPSTREAM=json.loads(_ref_bytes.decode('utf-8'))
REFERENCE=UPSTREAM['records']
assert len(REFERENCE)==48
assert Counter(x['budget'] for x in REFERENCE)=={128:24,256:24}
assert len(set(x['problem_id'] for x in REFERENCE if x['budget']==128))==24
assert all(x['status']=='ok' and int(x['nfe'])>0 for x in REFERENCE)
assert all(x['problem_id'] in {s['problem_id'] for s in SOURCES} for x in REFERENCE)
REFERENCE_LOOKUP={(x['problem_id'],int(x['budget'])):x for x in REFERENCE}
print('Reference provenance:',UPSTREAM['manifest'].get('config',{}).get('protocol','Notebook20'))
print('Earlier fixed-64-step generations:',len(REFERENCE),'and embedded SHA:',REFERENCE_SHA256)

### 6. Verified model loader, deterministic protocol, restart-safe record operations

In [ ]:
PROTOCOL='rsx21-mdlm-per-block32-plus-selfrepair-v1'
MODELS={
    'MDLM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1',
        sha='a284e895a6248256baf2f60502e54aba61b24c1a',
        actions=['tail25','tail50','middle25','global'],
        sampler_cls=MDLMSampler,config_cls=MDLMSamplerConfig,
    ),
    'BD3LM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1',
        sha='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=BD3LMSampler,config_cls=BD3LMSamplerConfig,
    ),
}
OUT_LEN=128; BLOCK=32; STEPS=128; TEMPERATURE=.15
GEN_SEED=38191; REPAIR_SEEDS=[38192,38193]; SELECT_MAX=8
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each supplied number once.'
CTX=None
SRC_MAP={s['problem_id']:s for s in SOURCES}
assert len(SRC_MAP)==24

def normalized_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,torch.Tensor):return x.detach().cpu().tolist()
    if hasattr(x,'item'):
        try:return x.item()
        except Exception:pass
    raise TypeError('Nonserializable '+repr(type(x)))

def atomic_json(path,value):
    path=Path(path);p=path.with_suffix(path.suffix+'.tmp')
    with p.open('w',encoding='utf-8') as f:
        json.dump(value,f,indent=2,ensure_ascii=False,default=normalized_json,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(p,path)

def append_record(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,default=normalized_json,allow_nan=False)+'\n')
        f.flush();os.fsync(f.fileno())

def completed_map(path,signature):
    records={}
    if not Path(path).exists():return records
    with open(path,encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except Exception as ex:raise ValueError(f'Malformed checkpoint JSONL {path}:{lineno}; backup before editing') from ex
            if r['signature']!=signature:raise ValueError('Checkpoint run-signature mismatch: '+str(path))
            if r['id'] in records:raise ValueError('Duplicate trial ID in checkpoint: '+str(r['id']))
            records[r['id']]=r
    return records

def model_dir(family):
    f=DRIVE/family
    f.mkdir(parents=True,exist_ok=True)
    return f

def config_signature(family):
    assert family=='MDLM', 'NB21 runs MDLM only.'
    m=MODELS[family]
    frozen=dict(protocol=PROTOCOL,family=family,model_id=m['repo'],model_sha=m['sha'],
                source_commit=SOURCE_COMMIT,source_payload_sha=PAYLOAD_SHA,
                n_sources=len(SOURCES),out_tokens=OUT_LEN,block=BLOCK,configured_steps_per_block=32,
                temperature=TEMPERATURE,gen_seed=GEN_SEED,repair_seeds=REPAIR_SEEDS,
                select_max=SELECT_MAX,actions=['tail25','tail50','middle25','global'],budgets=[128,256],fixed64_reference_sha=REFERENCE_SHA256)
    return hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),frozen

def files_for(family):
    folder=model_dir(family)
    return {'folder':folder,'manifest':folder/'run_manifest.json',
            'natural':folder/'generation_trials.jsonl',
            'repair':folder/'repair_trials.jsonl',
            'cohort':folder/'frozen_failure_ids.json',
            'errors':folder/'errors.jsonl'}

def init_family(family):
    paths=files_for(family);sig,cfg=config_signature(family)
    if paths['manifest'].exists():
        old=json.loads(paths['manifest'].read_text())
        if old['signature']!=sig:raise RuntimeError('Different protocol found for '+family+'; do not overwrite existing run. Rename Drive folder or restore previous notebook.')
    else:
        atomic_json(paths['manifest'],dict(signature=sig,config=cfg,
            created_utc=datetime.now(timezone.utc).isoformat(),gpu=GPU,torch_version=torch.__version__,transformers_version=transformers.__version__))
    return paths,sig

def unload_model():
    global CTX
    if CTX is not None:
        print('Unloading',CTX['family'])
        CTX['sampler']=None
        CTX['model']=None
        CTX['tokenizer']=None
        CTX=None
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_family(family):
    global CTX
    assert family in MODELS
    if CTX and CTX['family']==family:
        print(family,'already loaded');return
    unload_model()
    m=MODELS[family]; paths,sig=init_family(family)
    print('Loading',family,'public Hugging Face checkpoint (no token needed)',m['repo'])
    # HF cache-first; otherwise pinned full-snapshot download; simple retry.
    try:
        local=snapshot_download(repo_id=m['repo'],revision=m['sha'],local_files_only=True)
    except Exception:
        for trial in range(3):
            try:
                local=snapshot_download(repo_id=m['repo'],revision=m['sha'],max_workers=4)
                break
            except Exception:
                if trial==2:raise
                time.sleep(2**trial)
    local=Path(local)
    assert (local/'config.json').exists() and (local/'model.safetensors').exists()
    try:
        tokenizer=AutoTokenizer.from_pretrained(str(local),local_files_only=True,trust_remote_code=False,use_fast=True)
        if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
        if tokenizer.bos_token_id is None:tokenizer.bos_token=tokenizer.pad_token
        if tokenizer.mask_token_id is None:
            tokenizer.mask_token='<|mask|>'
        mask_id=int(tokenizer.mask_token_id)
        pad_id=int(tokenizer.pad_token_id)
        eos_id=int(tokenizer.eos_token_id)
        config=A2DQwen2Config.from_pretrained(str(local),local_files_only=True)
        if config.model_type!='a2d-qwen2':raise ValueError('Wrong checkpoint architecture: '+str(config.model_type))
        model=A2DQwen2LMHeadModel.from_pretrained(
            str(local),config=config,dtype=torch.float16,
            local_files_only=True,low_cpu_mem_usage=True,attn_implementation='sdpa',
        ).to('cuda').eval()
        nrows=int(model.get_input_embeddings().weight.shape[0])
        assert 0<=mask_id<nrows and mask_id not in [pad_id,eos_id],('Mask token missing from trained vocabulary',mask_id,nrows)
        sampler=m['sampler_cls'](model=model,tokenizer=tokenizer)
        CTX=dict(family=family,model=model,tokenizer=tokenizer,sampler=sampler,
                 mask=mask_id,eos=eos_id,pad=pad_id,paths=paths,signature=sig,
                 model_sha=m['sha'],model_id=m['repo'],classes=m)
        print('MODEL LOADED',family,'GPU allocated GB',round(torch.cuda.memory_allocated()/2**30,3),'mask id',mask_id)
    except Exception as exc:
        append_record(paths['errors'],dict(stage='model_load',signature=sig,error=repr(exc),traceback=traceback.format_exc(),timestamp=datetime.now(timezone.utc).isoformat()))
        raise

def prepare_ids(s):
    t=CTX['tokenizer']
    msg=[{'role':'system','content':'You solve quantitative reasoning tasks carefully.'},
         {'role':'user','content':s['prompt']+PROMPT_SUFFIX}]
    ids=t.apply_chat_template(msg,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    ids=[int(i) for i in ids]
    assert ids and CTX['mask'] not in ids
    return ids

def visible_answer_tokens(token_ids):
    result=[]
    for i in token_ids:
        if int(i) in [CTX['eos'],CTX['pad']]:break
        result.append(int(i))
    return result

def decode(token_ids):
    return CTX['tokenizer'].decode([int(i) for i in token_ids],skip_special_tokens=True).strip()

def sampler_config(answer_budget):
    # Fixed 32 steps for EACH 32-token block, NOT a fixed overall NFE.
    requested_steps=32*math.ceil(int(answer_budget)/BLOCK)
    a=dict(max_new_tokens=int(answer_budget),steps=requested_steps,block_size=BLOCK,
           temperature=TEMPERATURE,remasking='low_confidence',return_dict=True,
           right_shift_logits=False)
    if CTX['family']=='BD3LM':a['steps_per_block']=32
    return CTX['classes']['config_cls'](**a)

class CountCalls:
    def __init__(self,module):self.module=module;self.count=0;self.h=None
    def cb(self,*args):self.count+=1
    def __enter__(self):self.count=0;self.h=self.module.register_forward_pre_hook(self.cb);return self
    def __exit__(self,*args):self.h.remove();self.h=None

def set_seed(seed):
    torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed));random.seed(int(seed))

@torch.no_grad()
def real_generate(prompt,answer_budget,seed):
    set_seed(seed)
    t=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(prompt)],sampler_config(answer_budget),return_dict=True)
    raw=result.sequences[0].detach().cpu().tolist()
    if CTX['family']=='MDLM':
        assert raw[:len(prompt)]==list(prompt), 'MDLM changed observed prompt'
        answer=raw[len(prompt):]
    else:
        offset=math.ceil(len(prompt)/BLOCK)*BLOCK
        assert raw[offset-len(prompt):offset]==list(prompt),'BD3LM changed observed prompt'
        answer=raw[offset:]
    assert not any(t==CTX['mask'] for t in answer), 'Unresolved generation mask(s)'
    assert len(answer)<=answer_budget
    return dict(token_ids=visible_answer_tokens(answer),nfe=int(calls.count),
                raw_generated_tokens=len(answer),wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))

### A6a. Restore the user-completed NB21 48-generation checkpoint (safe, no GPU)
Runs after the original protocol/config signature is defined. **Never overwrites existing rows**; checks the exact signature and substantive model outputs.

In [ ]:
# NB21 rescue input: 48 user-executed GPU records, not simulated data.
import io,base64,zlib
NB21_EMBED_SHA='cc8c0996c1c53dc57e840808a1217d2890ba127642b587d02395f1d2a9dbd658'
NB21_PACK='...'
_doc=json.loads(zlib.decompress(base64.b64decode(NB21_PACK)))
_m=json.loads(base64.b64decode(_doc['run_manifest.json']))
_raw=base64.b64decode(_doc['generation_trials.jsonl'])
assert hashlib.sha256(_raw).hexdigest()==NB21_EMBED_SHA
assert _m['signature']==config_signature('MDLM')[0], 'Original user results are incompatible with this MDLM protocol; do not mix.'
_pp,_sig=init_family('MDLM')
_incoming=[json.loads(s) for s in _raw.decode().splitlines() if s.strip()]
assert len(_incoming)==48 and len(set(r['id'] for r in _incoming))==48
_present=completed_map(_pp['natural'],_sig)
def _scientific_r(r):
    return {k:v for k,v in r.items() if k not in ('wall_seconds','peak_vram_gb')}
_added=0
for r in _incoming:
    assert r['signature']==_sig and r['status']=='ok'
    if r['id'] in _present:
        if _scientific_r(_present[r['id']])!=_scientific_r(r):
            raise RuntimeError('Existing generation differs scientifically: '+r['id'])
    else:
        append_record(_pp['natural'],r);_added+=1
assert len(completed_map(_pp['natural'],_sig))==48
print('NB21 original generation records restored:',_added,'new rows; 48/48 available.')


### 7. Explicit MDLM CUDA smoke test (generation + native infill)

In [ ]:
load_family('MDLM')
source=SOURCES[0]
prompt=prepare_ids(source)
smoke=real_generate(prompt,32,45001)
assert len(smoke['token_ids'])>=4 and smoke['nfe']>0
# A real infill must leave observed prompt and selected answer tokens unchanged.
original=smoke['token_ids']; old=list(original);canvas=[CTX['mask'] if k>=len(old)//2 else old[k] for k in range(len(old))]
steps_infill=32*math.ceil((len(prompt)+len(canvas))/BLOCK)
cfg=MDLMSamplerConfig(steps=steps_infill,block_size=BLOCK,temperature=.15,remasking='low_confidence',return_dict=True)
with CountCalls(CTX['model']) as call_audit:
    result=CTX['sampler'].infill([prompt+canvas],cfg,return_dict=True)
seq=result.sequences[0].detach().cpu().tolist()
assert seq[:len(prompt)]==prompt
assert seq[len(prompt):len(prompt)+len(old)//2]==old[:len(old)//2]
assert CTX['mask'] not in seq
print('REAL CUDA generation + native infill SMOKE PASS. Forward calls:',smoke['nfe'],call_audit.count)

### 8. Cost-controlled generation code and persistent batch runner

In [ ]:
BUDGETS=(128,256)
assert CTX is not None and CTX['family']=='MDLM'

def generation_trial(s,budget):
    ret=real_generate(prepare_ids(s),int(budget),GEN_SEED)
    decoded=decode(ret['token_ids'])
    v=verify(decoded,s)
    return dict(id=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}",problem_id=s['problem_id'],
       dataset=s['dataset'],target=float(s['target']),prompt=s['prompt'],budget=int(budget),
       generation_seed=GEN_SEED,output=decoded,verification=v,
       task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
       numeric_success=bool(v['numeric_success']), visible_answer_tokens=len(ret['token_ids']),
       step_policy='32_per_32token_block',configured_total_steps=32*math.ceil(budget/BLOCK),
       budget_exhaustion_proxy=bool(ret['raw_generated_tokens']==budget and len(ret['token_ids'])>=budget-2),**ret)

def gen_progress():
    done=completed_map(CTX['paths']['natural'],CTX['signature'])
    for budget in BUDGETS:
        ready=sum(int(x['budget']==budget and x['status']=='ok') for x in done.values())
        print(f'Controlled generation {budget}: {ready}/24 completed')

def generation_batch(budget,start,n=4,max_minutes=8):
    assert CTX is not None and CTX['family']=='MDLM'
    assert int(budget) in BUDGETS and 0<=start<24 and 1<=n<=4 and start+n<=24
    path=CTX['paths']['natural'];sig=CTX['signature'];done=completed_map(path,sig)
    began=time.monotonic()
    for s in SOURCES[start:start+n]:
        tid=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}"
        if tid in done:
            assert done[tid]['status']=='ok'
            print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Reached soft cutoff. Rerun this batch to resume.');break
        try:
            r=generation_trial(s,budget);r.update(signature=sig,status='ok')
            append_record(path,r)
            print('SAVED',tid,'task=',r['task_success'],'NFEs=',r['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='generation',id=tid,error=repr(exc),
                  traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Generation failed. Existing Drive checkpoints are safe. Check errors.jsonl.') from exc
    gen_progress()

### 9. 128-token MDLM: 6 short batches × 4 generations

In [ ]:
generation_batch(128,0,n=4,max_minutes=8)

In [ ]:
generation_batch(128,4,n=4,max_minutes=8)

In [ ]:
generation_batch(128,8,n=4,max_minutes=8)

In [ ]:
generation_batch(128,12,n=4,max_minutes=8)

In [ ]:
generation_batch(128,16,n=4,max_minutes=8)

In [ ]:
generation_batch(128,20,n=4,max_minutes=8)

### 10. 256-token MDLM: 6 short batches × 4 generations

In [ ]:
generation_batch(256,0,n=4,max_minutes=8)

In [ ]:
generation_batch(256,4,n=4,max_minutes=8)

In [ ]:
generation_batch(256,8,n=4,max_minutes=8)

In [ ]:
generation_batch(256,12,n=4,max_minutes=8)

In [ ]:
generation_batch(256,16,n=4,max_minutes=8)

In [ ]:
generation_batch(256,20,n=4,max_minutes=8)

### 10a. Safe intermediate download: 48 generation results (before any repairs)

In [ ]:
# Export the entire generation stage separately before proceeding to repairs.
assert CTX and CTX['family']=='MDLM'
_checkpoint=completed_map(CTX['paths']['natural'],CTX['signature'])
_expected={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
assert set(_checkpoint)==_expected, f'Generation stage incomplete: {len(_expected-set(_checkpoint))} missing'
assert all(x['status']=='ok' for x in _checkpoint.values())
_export_zip=Path('/content/RepairScopeX_NB21_MDLM_ConstantPerBlock_Generation_only.zip')
with zipfile.ZipFile(_export_zip,'w',compression=zipfile.ZIP_DEFLATED) as _z:
    _z.write(CTX['paths']['manifest'],CTX['paths']['manifest'].name)
    _z.write(CTX['paths']['natural'],CTX['paths']['natural'].name)
print('Generation-only backup verified:',_export_zip)
from google.colab import files
files.download(str(_export_zip))

### 11. Freeze or safely resume eight 128-token failures (v2 metadata-only fix)

This replaces the raw generation-file checksum check with a scientifically pinned 48-record fingerprint. It accepts only the two previously audited timing-only ZIP variants, and otherwise refuses inconsistent frozen cohorts.


In [ ]:
# NB21 v2: Cohort selection using a content fingerprint, not GPU timing.
# The original and the new generation-only ZIPs were compared field-by-field.
# Only wall_seconds changed. This fingerprint pins every scientific field of all 48 records.
from collections import defaultdict, Counter

# Audited against BOTH supplied Notebook-21 generation archives.
AUDITED_SCIENTIFIC_SHA256 = "8c6abe099a1c5f9f8932f44f35f0929c28545398a607e626e90a7474ec29cd90"
AUDITED_LEGACY_FILE_SHA256 = {
    "1c9a2e85c99622bf35c1813fabc481d3241d9d94f9608c55b4603262a94fcc39",
    "cc8c0996c1c53dc57e840808a1217d2890ba127642b587d02395f1d2a9dbd658",
}

def scientific_generation_digest(rows):
    # Exclude only execution telemetry, which has no bearing on outputs or cohort.
    normalized = [{k: v for k, v in rec.items()
                   if k not in ('wall_seconds', 'peak_vram_gb')}
                  for rec in rows]
    normalized.sort(key=lambda rec: rec['id'])
    body = json.dumps(normalized, sort_keys=True, ensure_ascii=False,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(body).hexdigest()

def choose_failures():
    assert CTX is not None and CTX['family'] == 'MDLM'
    p = CTX['paths']; sig = CTX['signature']
    gens = completed_map(p['natural'], sig)
    expected = {f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}"
                for s in SOURCES for b in BUDGETS}
    assert set(gens) == expected, f"Generation checkpoint not complete: {len(expected-set(gens))} missing"
    assert len(gens) == 48
    assert all(x['status'] == 'ok' and x['signature'] == sig for x in gens.values())
    digest = scientific_generation_digest(list(gens.values()))
    if digest != AUDITED_SCIENTIFIC_SHA256:
        raise RuntimeError('Scientific generation records differ from the two audited NB21 ZIPs. '
                           'Do not reuse a frozen cohort: investigate changes in predictions or labels.')

    pool = defaultdict(list)
    for x in gens.values():
        if x['budget'] == 128 and not x['task_success'] and len(x['token_ids']) >= 16:
            # Revalidate the failure before selecting it.
            assert not verify(x['output'], SRC_MAP[x['problem_id']])['task_success']
            pool[x['dataset']].append(x)
    quotas = [('countdown_fresh', 3), ('gsm8k', 3), ('svamp', 2)]
    assert all(len(pool[k]) >= n for k,n in quotas), 'Not enough eligible failures for predeclared dataset quotas.'
    selected = []
    for k,n in quotas:
        ordered = sorted(pool[k], key=lambda x:hashlib.sha256(
            ('rsx21cohort:'+x['problem_id']).encode()).hexdigest())
        selected.extend(ordered[:n])
    ids=[x['problem_id'] for x in selected]
    assert len(ids)==len(set(ids))==8
    new_content = {
        'source_generation_sha256': hashlib.sha256(p['natural'].read_bytes()).hexdigest(),
        'generation_signature': sig,
        'ids': ids,
        'budget': 128,
        'dataset_counts': dict(Counter(x['dataset'] for x in selected)),
        'scientific_generation_sha256': digest,
    }

    if p['cohort'].exists():
        current = json.loads(p['cohort'].read_text())
        fields = ('generation_signature','ids','budget','dataset_counts')
        for field in fields:
            if current.get(field) != new_content[field]:
                raise RuntimeError(f"Frozen cohort changed in {field}; existing checkpoint preserved.")
        prev_science = current.get('scientific_generation_sha256')
        if prev_science is not None and prev_science != digest:
            raise RuntimeError('Previously frozen scientific content differs; checkpoint preserved.')
        prior_file_sha = current.get('source_generation_sha256')
        if prior_file_sha != new_content['source_generation_sha256']:
            # Only accept metadata-only cases covered by the TWO source archives audited
            # before constructing this notebook. Do not silently accept any other hash.
            if prior_file_sha not in AUDITED_LEGACY_FILE_SHA256:
                raise RuntimeError('Stored frozen cohort refers to an unreviewed generation file checksum. '
                                   'Supply that older checkpoint for an explicit comparison; nothing was overwritten.')
            print('Accepted previously audited timing-only generation change; frozen cohort is unchanged.')
        # Never overwrite a frozen cohort record to make an error disappear.
    else:
        atomic_json(p['cohort'], new_content)
        print('Created new frozen failure cohort (scientific fingerprint verified).')

    # Important if resuming after repairs have already begun.
    repairs=completed_map(p['repair'],sig)
    allowed={f"{f['problem_id']}|{a}|{seed}" for f in selected
             for a in ('tail25','tail50','middle25','global')
             for seed in REPAIR_SEEDS}
    if set(repairs)-allowed:
        raise RuntimeError('Saved repair trials refer to a different failure cohort. Stop and inspect.')
    selected_by_id={f['problem_id']:f for f in selected}
    for rid,repair in repairs.items():
        failure=selected_by_id[repair['problem_id']]
        if repair.get('original_output') != failure['output']:
            raise RuntimeError(f'Repair origin mismatch for {rid}; existing repairs preserved.')
    print('FROZEN: 8 genuine native failures',new_content['dataset_counts'],
          '| generation SHA256:',digest[:16], '| existing repairs:',len(repairs))
    return selected

SELECTED_FAILURES=choose_failures()


### 12. Native MDLM masked-token repair, preservation audit and compute accounting

In [ ]:
ACTIONS=('tail25','tail50','middle25','global')
REPAIR_SEEDS=(38192,38193)

def repair_canvas(old,action,mask):
    n=len(old)
    assert n>=16
    if action=='global': changed=set(range(n))
    elif action=='tail25': changed=set(range(max(1,int(.75*n)),n))
    elif action=='tail50': changed=set(range(max(1,int(.50*n)),n))
    elif action=='middle25':
        width=max(1,math.ceil(.25*n));start=(n-width)//2
        changed=set(range(start,start+width))
    else: raise ValueError(action)
    assert len(changed)>0
    locked=set(range(n))-changed
    return [mask if k in changed else int(old[k]) for k in range(n)],sorted(locked), sorted(changed)

@torch.no_grad()
def self_repair(failure,action,seed):
    s=SRC_MAP[failure['problem_id']]
    old=[int(t) for t in failure['token_ids']]
    prompt=prepare_ids(s)
    canvas,locked,changed=repair_canvas(old,action,CTX['mask'])
    assert all(canvas[i]==old[i] for i in locked)
    assert all(canvas[i]==CTX['mask'] for i in changed)
    # MDLM.infIll divides steps over ALL blocks, including observed prompt blocks.
    # Give every 32-token input block 32 requested denoising steps; empty blocks skip.
    total_steps=32*math.ceil((len(prompt)+len(canvas))/BLOCK)
    cfg=MDLMSamplerConfig(steps=total_steps,block_size=BLOCK,temperature=TEMPERATURE,
        remasking='low_confidence',return_dict=True,right_shift_logits=False)
    set_seed(seed);ts=time.perf_counter();torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        out=CTX['sampler'].infill([prompt+canvas],cfg,return_dict=True)
    seq=[int(t) for t in out.sequences[0].detach().cpu().tolist()]
    assert seq[:len(prompt)]==prompt,'Observed question altered by MDLM infill.'
    repaired=seq[len(prompt):]
    assert len(repaired)==len(old), 'Repaired canvas length changed'
    assert all(repaired[i]==old[i] for i in locked),'Frozen tokens were modified'
    assert CTX['mask'] not in repaired,'Unfilled masks remain'
    shown=visible_answer_tokens(repaired)
    text=decode(shown)
    v=verify(text,s)
    original_visible=visible_answer_tokens(old)
    changes=sum(int(a!=b) for a,b in zip(repaired,old))
    preserve=1.0-changes/len(old)
    same_numeric=lastnum(text)==lastnum(failure['output']) if lastnum(text) is not None else False
    old_task_ok=verify(failure['output'],s)['task_success']
    assert not old_task_ok,'Starting point must be a task failure.'
    return dict(id=f"{failure['problem_id']}|{action}|{seed}",problem_id=failure['problem_id'],
         dataset=s['dataset'],action=action,seed=int(seed),original_output=failure['output'],
         repaired_output=text,original_token_count=len(old),num_reopened=len(changed),num_locked=len(locked),
         preservation=float(preserve),locked_tokens_exact=True,changed_token_positions=changes,
         wrong_answer_recurrence_numeric=bool(not v['task_success'] and same_numeric),
         exact_wrong_output_recurrence=bool(not v['task_success'] and repaired==old),
         task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
         numeric_success=bool(v['numeric_success']),verification=v,
         native_method='MDLMSampler.infill',requested_steps=int(total_steps),nfe=int(calls.count),
         wall_seconds=float(time.perf_counter()-ts),peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30),
         raw_repaired_tokens=repaired)

REPAIR_JOBS=[(s,act,seed) for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS]
assert len(REPAIR_JOBS)==64

def repair_progress():
    d=completed_map(CTX['paths']['repair'],CTX['signature'])
    print('Saved repair trials:',len(d),'/64')

def repair_batch(start,n=4,max_minutes=8):
    assert CTX is not None and CTX['family']=='MDLM'
    assert 0<=start<64 and 1<=n<=4 and start+n<=64
    jobs=REPAIR_JOBS[start:start+n];path=CTX['paths']['repair'];sig=CTX['signature']
    done=completed_map(path,sig)
    began=time.monotonic()
    for f,action,seed in jobs:
        tid=f"{f['problem_id']}|{action}|{seed}"
        if tid in done:
            assert done[tid]['status']=='ok'; print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Soft 8-minute cutoff reached. Rerun cell to complete this batch.');break
        try:
            row=self_repair(f,action,seed);row.update(signature=sig,status='ok')
            append_record(path,row)
            print('SAVED',tid,'success=',row['task_success'],'NFEs=',row['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='self_repair',id=tid,error=repr(exc),
                traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Self-repair failed, previous trials safe on Drive. Check errors.jsonl.') from exc
    repair_progress()

### 13. Repairs 01–32 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(0,n=4,max_minutes=8)

In [ ]:
repair_batch(4,n=4,max_minutes=8)

In [ ]:
repair_batch(8,n=4,max_minutes=8)

In [ ]:
repair_batch(12,n=4,max_minutes=8)

In [ ]:
repair_batch(16,n=4,max_minutes=8)

In [ ]:
repair_batch(20,n=4,max_minutes=8)

In [ ]:
repair_batch(24,n=4,max_minutes=8)

In [ ]:
repair_batch(28,n=4,max_minutes=8)

### 14. Repairs 33–64 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(32,n=4,max_minutes=8)

In [ ]:
repair_batch(36,n=4,max_minutes=8)

In [ ]:
repair_batch(40,n=4,max_minutes=8)

In [ ]:
repair_batch(44,n=4,max_minutes=8)

In [ ]:
repair_batch(48,n=4,max_minutes=8)

In [ ]:
repair_batch(52,n=4,max_minutes=8)

In [ ]:
repair_batch(56,n=4,max_minutes=8)

In [ ]:
repair_batch(60,n=4,max_minutes=8)

### 15. Integrity checks, paired length comparisons, clustered repair summaries

In [ ]:
import csv

def write_csv(file,rows):
    if not rows:return
    keys=list(rows[0].keys())
    with open(file,'w',newline='',encoding='utf-8') as h:
        w=csv.DictWriter(h,fieldnames=keys,extrasaction='ignore');w.writeheader();w.writerows(rows)

def wilson95(k,n):
    z=1.95996398454;p=k/n;scale=1+z*z/n
    a=(p+z*z/(2*n))/scale;b=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/scale
    return max(0,a-b),min(1,a+b)

def paired_exact(a,b):
    n01=sum((not bool(x)) and bool(y) for x,y in zip(a,b))
    n10=sum(bool(x) and (not bool(y)) for x,y in zip(a,b))
    discord=n01+n10
    p=1.0 if discord==0 else min(1.0,2*sum(math.comb(discord,k) for k in range(min(n01,n10)+1))/2**discord)
    return {'new_only':n01,'old_only':n10,'discordant':discord,'two_sided_exact_p':p}

def finish_experiment():
    assert CTX and CTX['family']=='MDLM'
    p=CTX['paths'];sig=CTX['signature'];g=completed_map(p['natural'],sig);r=completed_map(p['repair'],sig)
    expected_g={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
    expected_r={f"{s['problem_id']}|{act}|{seed}" for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS}
    assert set(g)==expected_g and len(g)==48, f'Incomplete generations: missing {len(expected_g-set(g))}'
    assert set(r)==expected_r and len(r)==64, f'Incomplete repairs: missing {len(expected_r-set(r))}'
    assert all(x['status']=='ok' for x in list(g.values())+list(r.values()))
    assert all(x['locked_tokens_exact'] for x in r.values())
    generations=sorted(g.values(),key=lambda x:(x['budget'],x['problem_id']))
    repairs=sorted(r.values(),key=lambda x:(x['problem_id'],x['action'],x['seed']))
    generation_summary=[];paired=[]
    ids=[s['problem_id'] for s in SOURCES]
    for budget in BUDGETS:
        new={x['problem_id']:x for x in generations if x['budget']==budget}
        old={x['problem_id']:x for x in REFERENCE if x['budget']==budget}
        assert set(new)==set(old)==set(ids)
        nh=sum(bool(x['task_success']) for x in new.values());oh=sum(bool(x['task_success']) for x in old.values())
        lo,hi=wilson95(nh,24)
        generation_summary.append(dict(budget=budget,conditions='fixed32_per_block_vs_fixed_total64',n=24,
             new_successes=nh,old_successes=oh,new_task_rate=nh/24,old_task_rate=oh/24,
             wilson95_low=lo,wilson95_high=hi,new_mean_nfe=sum(x['nfe'] for x in new.values())/24,
             old_mean_nfe=sum(x['nfe'] for x in old.values())/24,
             new_mean_seconds=sum(x['wall_seconds'] for x in new.values())/24,
             old_mean_seconds=sum(x['wall_seconds'] for x in old.values())/24))
        paired.append(dict(budget=budget,**paired_exact([old[i]['task_success'] for i in ids],
                                            [new[i]['task_success'] for i in ids])))
    repair_summary=[]
    for act in ACTIONS:
        group=[x for x in repairs if x['action']==act]
        assert len(group)==16
        success=sum(int(x['task_success']) for x in group)
        # Cluster 2 repeated trials by original failure, report problem-level observed success.
        best_by_problem={pid:any(x['task_success'] for x in group if x['problem_id']==pid) for pid in {x['problem_id'] for x in group}}
        p_success=sum(best_by_problem.values());lo,hi=wilson95(p_success,8)
        repair_summary.append(dict(action=act,trials=len(group),unique_failed_problems=8,successful_trials=success,
            trial_success_rate=success/16,problems_ever_repaired=p_success,
            problem_ever_repair_rate=p_success/8,wilson95_problem_low=lo,wilson95_problem_high=hi,
            mean_nfe=sum(x['nfe'] for x in group)/len(group),
            mean_preservation=sum(x['preservation'] for x in group)/len(group),
            mean_wall_seconds=sum(x['wall_seconds'] for x in group)/len(group),
            same_wrong_numeric_given_failure=sum(int(x['wrong_answer_recurrence_numeric']) for x in group if not x['task_success'])/
                 max(1,sum(int(not x['task_success']) for x in group))))
    out=p['folder']
    write_csv(out/'controlled_length_summary.csv',generation_summary)
    write_csv(out/'paired_fixed_total_vs_per_block.csv',paired)
    write_csv(out/'repair_action_summary.csv',repair_summary)
    oracle=sum(int(any(x['task_success'] for x in repairs if x['problem_id']==sid)) for sid in [x['problem_id'] for x in SELECTED_FAILURES])/8
    best_fixed=max(x['trial_success_rate'] for x in repair_summary)
    gate=dict(run_signature=sig,expected_natural_generations=48,actual_natural_generations=len(g),
         expected_repair_trials=64,actual_repair_trials=len(r),
         unique_repair_problems=8,all_frozen_positions_audited=True,
         native_method='official_mdlm_infill',generation_policy='32_requested_steps_per_32_token_block',
         positive_repair_trials=sum(int(x['task_success']) for x in repairs),
         best_fixed_trial_success=best_fixed,in_sample_oracle_any_seed_success=oracle,
         signal_ready=bool(sum(int(x['task_success']) for x in repairs)>0),complete=True)
    atomic_json(out/'NB21_GATE.json',gate)
    artifacts=[p['manifest'],p['natural'],p['repair'],p['cohort'],out/'controlled_length_summary.csv',
      out/'paired_fixed_total_vs_per_block.csv',out/'repair_action_summary.csv',out/'NB21_GATE.json']
    hashes={x.name:hashlib.sha256(x.read_bytes()).hexdigest() for x in artifacts}
    atomic_json(out/'artifact_manifest.json',dict(sha256=hashes,upstream_reference_sha256=REFERENCE_SHA256))
    artifacts.append(out/'artifact_manifest.json')
    zip_file=Path('/content/RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip')
    with zipfile.ZipFile(zip_file,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in artifacts:z.write(f,f.name)
    print('NB21 COMPLETE. Gate:',json.dumps(gate,indent=2))
    print('Generation:',generation_summary, chr(10), 'Repair:',repair_summary, chr(10), 'ZIP:',zip_file)
    return zip_file

RESULT_ZIP=finish_experiment()
from google.colab import files
files.download(str(RESULT_ZIP))

### 16. Next step after running

Return `RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip` for analysis. If there are still no repairs, do not repeat this checkpoint again without changing the research question. Finish the pending BD3LM length experiment, then compare models on the original paired problems with an explicit compute policy.

A signal gate of `false` is a valid negative finding; it is not a reason to discard the run. This notebook reports verified token-preservation but does not claim to have verified the semantic correctness of all word-problem reasoning chains.

### A16. Persist the completed original NB21 ZIP for Phase B
Run **only after** `finish_experiment()` completes. The final NB29 archive will include this ZIP if present.

In [ ]:
# This cell executes AFTER the NB21 experiment has passed its 64/64 repair gate.
assert RESULT_ZIP.is_file()
NB29_SHARED=Path('/content/drive/MyDrive/RepairScopeX/RSX29_Combined_Outputs')
NB29_SHARED.mkdir(parents=True,exist_ok=True)
NB21_PERSIST=NB29_SHARED/'RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip'
if NB21_PERSIST.exists():
    assert hashlib.sha256(NB21_PERSIST.read_bytes()).hexdigest()==hashlib.sha256(RESULT_ZIP.read_bytes()).hexdigest(), 'Existing final NB21 bundle differs; preserve and investigate before replacing.'
else:shutil.copy2(RESULT_ZIP,NB21_PERSIST)
print('NB21 COMPLETE, saved permanently to:',NB21_PERSIST)
print('MANDATORY NEXT: Colab Runtime > Restart session; then run ONLY Phase B cells below.')

---

# PHASE B — Notebook 29: Prospective semantic-error repair

## MANDATORY RUNTIME RESTART

**STOP HERE.** In Colab choose **Runtime → Restart session**, keep the T4 GPU, and then run **only Phase B cells below**. The two upstream source stacks use different pinned Transformers interfaces. Do not run `Run all` from the beginning after restarting. Resume completed NB29 trials from the dedicated Drive directory. This is a *single notebook containing two separate phases*, not an attempt to keep incompatible models resident together.

### Prespecified design

- Frozen synthetic problems with **new seeds disjoint** from Notebook 28. No use of Notebook 28 repair outcomes for calibration.
- Model-native generation with the ALoDLM-1.7B FP32 official decoder, using the proven upstream commit and model revision.
- Semantic-error eligibility: **unambiguously wrong extracted final numeric answer** (`FINAL_ANSWER`, `\boxed`, or explicit answer phrase), or an inspectable illegal Countdown expression with the original numbers. Correct boxed answers, missing answers, and ambiguous/conflicting answers are NOT eligible.
- Frozen target cohort: 12 calibration failures (4 of each task) and 24 held-out failures (8 of each). If a task's target cannot be met, report the deficit, export a partial archive, and continue on a later Colab session. **Never replace unavailable semantic failures with format-only failures.**
- Calibration: 12×4 scopes×2 seeds = **96** model-native repair calls.
- Held-out: 24×4 scopes×3 seeds = **288 unique candidates**; five predeclared policy decisions per problem×seed share these candidates when identical.
- Additional two-stage strategy: 24×2 policies×2 seeds = **96 logical evaluations**, requiring up to 192 GPU proposals (checkpointed, may stop early).
- The primary outcome is recovery of **distinct semantic-failure problems**; also report per-trial success, compute, recurrence, and format success. No gold answer is used in policy selection or stopping.

*Runtime budget:* designed for a few hours **across resumable Colab sessions**, not guaranteed to finish in one three-hour session. Strictly check the real-GPU smoke test and first successful cache write before continuing.


### B0. Official ALoDLM source / dependencies — run only AFTER runtime restart

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX29'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned official ALoDLM source:',CODE_COMMIT)

### B1. Real T4 GPU, new frozen protocol, and append-only Drive storage

In [ ]:
import os, sys, gc, json, csv, re, math, ast, time, random, hashlib, zipfile, tempfile, statistics, shutil, platform, traceback, subprocess
from datetime import datetime,timezone
from pathlib import Path
from collections import defaultdict,Counter
from dataclasses import dataclass
import torch, numpy as np, pandas as pd, transformers
from scipy.stats import binomtest, beta
from huggingface_hub import snapshot_download
from types import SimpleNamespace
from google.colab import drive, files
import matplotlib.pyplot as plt
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('REQUIRES a real GPU: Colab > Runtime > Change runtime type > T4 GPU.')
if torch.cuda.get_device_properties(0).total_memory < 12*2**30: raise RuntimeError('Needs at least 12 GiB GPU VRAM.')
if torch.cuda.memory_allocated() > 2*2**30: raise RuntimeError('Restart runtime: unexpected other GPU allocations.')
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RepairScopeX/NB29_SemanticError_FP32_V1')
ROOT.mkdir(parents=True,exist_ok=True)
ART=ROOT/'artifacts'; ART.mkdir(exist_ok=True)
CACHE=ROOT/'records'; CACHE.mkdir(exist_ok=True)
MAX_SESSION_MINUTES=160
SESSION_START=time.monotonic()
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
PRECISION='fp32'; Q=0.4; TAU=0.4; WINDOW=16; MAX_NEW=96
GEN_MODE='left1'; REPAIR_MODE='entropy'
BASE_SEED=992941
CAL_TARGET={'countdown':4,'arithmetic':4,'multistep':4}
TEST_TARGET={'countdown':8,'arithmetic':8,'multistep':8}
ACTIONS=('global','tail25','tail50','tail75')
CAL_SEEDS=(23001,23002)
TEST_SEEDS=(24001,24002,24003)
STAGED_SEEDS=(25001,25002)
ABLATION_SEEDS=(26001,26002)
POLICIES=('global','best_fixed','adaptive','random_scope','tail50')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Torch',torch.__version__,'Transformers',transformers.__version__)

def canonical(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'),default=str)
def sha(v): return hashlib.sha256((v if isinstance(v,bytes) else canonical(v).encode())).hexdigest()
def atomic_write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',encoding='utf8') as h:
        h.write(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,default=str)+'\n');h.flush();os.fsync(h.fileno())
    os.replace(tmp,path)
def rec_path(stage,tid):return CACHE/stage/(sha(tid)+'.json')
def save_rec(stage,rec):
    path=rec_path(stage,rec['trial_id'])
    if path.exists():
        old=json.loads(path.read_text());
        if sha(old)!=sha(rec):raise RuntimeError('Refusing to overwrite a different completed record: '+rec['trial_id'])
        return
    atomic_write(path,rec)
def get_rec(stage,tid):
    p=rec_path(stage,tid)
    if not p.exists():return None
    r=json.loads(p.read_text())
    if r['trial_id']!=tid or r.get('protocol_hash')!=RUN_SIG:raise RuntimeError('Checkpoint signature or identity mismatch '+tid)
    return r
def all_recs(stage):
    out=[]
    for p in sorted((CACHE/stage).glob('*.json')):
        r=json.loads(p.read_text())
        assert r['protocol_hash']==RUN_SIG
        out.append(r)
    assert len({r['trial_id'] for r in out})==len(out)
    return out
def stop_if_near_budget():
    used=(time.monotonic()-SESSION_START)/60
    if used >= MAX_SESSION_MINUTES:raise RuntimeError(f'Session wall-time safety stop ({used:.1f}/{MAX_SESSION_MINUTES} min). Export partial ZIP and resume in a new Colab session; earlier records are saved.')

PROTOCOL={'version':'RSX29-AloDLM-semantic-wrong-v1','model':MODEL_ID,'model_rev':MODEL_REV,'code_commit':CODE_COMMIT,
          'fp':PRECISION,'q':Q,'tau':TAU,'window':WINDOW,'max_new':MAX_NEW,
          'gen_mode':GEN_MODE,'repair_mode':REPAIR_MODE,'cal_target':CAL_TARGET,'test_target':TEST_TARGET,
          'actions':ACTIONS,'cal_seeds':CAL_SEEDS,'test_seeds':TEST_SEEDS,
          'staged_seeds':STAGED_SEEDS,'ablation_seeds':ABLATION_SEEDS,'base_seed':BASE_SEED}
RUN_SIG=sha(PROTOCOL)
manifest=ROOT/'protocol.json'
if manifest.exists():
    existing=json.loads(manifest.read_text())
    if existing['hash']!=RUN_SIG or sha(existing['protocol'])!=sha(PROTOCOL):raise RuntimeError('Different experiment in this Drive path. NEVER overwrite or mix protocols. Use a new folder.')
else:atomic_write(manifest,{'hash':RUN_SIG,'protocol':PROTOCOL,'created_utc':datetime.now(timezone.utc).isoformat()})

### B2. Disjoint, frozen deterministic benchmark (no model-generated gold labels)

In [ ]:
def make_problem(kind, rng, i, split):
    # Derived from deterministic synthetic templates, not GSM8K or SVAMP. No LLM-generated labels.
    if kind=='countdown':
        a,b,c=[rng.randint(3,30) for _ in range(3)]
        op=rng.choice(('plusminus','sum','prodminus','twice'))
        if op=='plusminus':target=a+b-c
        elif op=='sum':target=a+b+c
        elif op=='prodminus':target=a*b-c
        else:target=a*(b-c)
        prompt=f'Use the numbers {a}, {b}, and {c} exactly once to produce the target {target}. Return a valid arithmetic expression and end with `FINAL_ANSWER: <number>`.'
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[a,b,c],'template':op}
    if kind=='arithmetic':
        mode=i%4
        a=rng.randint(18,100);b=rng.randint(4,30)
        if mode==0:
            target=a+b;prompt=f'A collection contains {a} coins. A friend adds {b} more coins. How many coins are there now? End with FINAL_ANSWER: <number>.'
        elif mode==1:
            target=a-b;prompt=f'A shelf holds {a} books. Exactly {b} books are borrowed. How many books remain? End with FINAL_ANSWER: <number>.'
        elif mode==2:
            target=a*b;prompt=f'There are {a} identical boxes, each containing {b} pens. How many pens are there altogether? End with FINAL_ANSWER: <number>.'
        else:
            target=a;prompt=f'A number of stickers is divided equally among {b} children, and every child receives {a} stickers. How many stickers were there originally? End with FINAL_ANSWER: <number>.';target=a*b
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}
    mode=i%4
    if mode==0:
        a=rng.randint(20,90);b=rng.randint(3,10);c=rng.randint(5,24);d=rng.randint(1,18)
        target=a+b*c-d;prompt=f'A warehouse begins with {a} packages. It receives {b} cartons with {c} packages each, then ships {d} packages. How many packages remain? End with FINAL_ANSWER: <number>.'
    elif mode==1:
        a=rng.randint(5,18);b=rng.randint(4,17);c=rng.randint(2,10);d=rng.randint(1,10)
        target=a*b+c*d;prompt=f'A school buys {a} packets containing {b} pencils each and {c} packets containing {d} pencils each. How many pencils in total? End with FINAL_ANSWER: <number>.'
    elif mode==2:
        a=rng.randint(3,12);b=rng.randint(5,20);c=rng.randint(3,25)
        target=a*b-c;prompt=f'A gardener plants {a} rows with {b} flowers each. Later {c} flowers are removed. How many flowers remain? End with FINAL_ANSWER: <number>.'
    else:
        a=rng.randint(10,40);b=rng.randint(2,8);c=rng.randint(5,20)
        target=a*b+c;prompt=f'A shop has {a} cartons of {b} bottles. A delivery adds {c} single bottles. How many bottles are available? End with FINAL_ANSWER: <number>.'
    return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}

GLOBAL_FINGERPRINTS=set()
def build_bank(split,n_per_kind):
    out=[];seen=set()
    for k_idx,kind in enumerate(('countdown','arithmetic','multistep')):
        for i in range(n_per_kind):
            for retry in range(200):
                seed=BASE_SEED+(0 if split=='calibration' else 1000000)+k_idx*10000+i*11+retry*310009
                rng=random.Random(seed)
                d=make_problem(kind,rng,i,split)
                fp=sha({'prompt':d['prompt'],'kind':kind})
                if fp not in seen and fp not in GLOBAL_FINGERPRINTS:
                    seen.add(fp);GLOBAL_FINGERPRINTS.add(fp)
                    d.update(split=split,problem_id=f'rsx29:{split}:{kind}:{i:04d}',
                             problem_fingerprint=fp,collision_retry=retry)
                    out.append(d)
                    break
            else:raise RuntimeError('Could not generate distinct frozen task within 200 deterministic retries.')
    return out
CAL_BANK=build_bank('calibration',88)
TEST_BANK=build_bank('heldout',128)
assert len(set(r['problem_fingerprint'] for r in CAL_BANK+TEST_BANK))==len(CAL_BANK+TEST_BANK)
BANK_HASH=sha(CAL_BANK+TEST_BANK)
BANK_PATH=ROOT/'frozen_problem_bank.json'
if BANK_PATH.exists():assert json.loads(BANK_PATH.read_text())['hash']==BANK_HASH
else:atomic_write(BANK_PATH,{'hash':BANK_HASH,'problems':CAL_BANK+TEST_BANK})
print('Frozen',len(CAL_BANK),'calibration candidates and',len(TEST_BANK),'heldout candidates; checksum',BANK_HASH)

### B3. STRICT semantic-error eligibility and answer-format separation

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')


### B4. Load official ALoDLM FP32 model; real CUDA smoke test

In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
assert MODEL_REV in str(MODEL_DIR) or Path(MODEL_DIR).is_dir()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
decoder=load_decoder(args)
model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Model loaded FP32; CUDA allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
PROMPT_SUFFIX='\n\nSolve carefully. For Countdown, show an arithmetic expression using all given numbers exactly once. End your response with FINAL_ANSWER: <number>.'
def encode_prompt(p):
    return list(tokenizer.apply_chat_template([{'role':'user','content':p['prompt']+PROMPT_SUFFIX}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def decode_ids(ids):return tokenizer.decode(ids,skip_special_tokens=True).strip()
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
@torch.inference_mode()
def invoke(prompt_ids,new_tokens,mode,seed):
    assert new_tokens>0 and new_tokens<=MAX_NEW
    stop_if_near_budget();set_seed(int(seed))
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(new_tokens),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter();result=decoder.generate(list(prompt_ids),cfg)
    ids=list(map(int,result['token_ids']))
    return {'token_ids':ids,'generated_tokens':int(result.get('generated_tokens',len(ids))),
            'outer_steps':int(result['outer_steps']),'recurrent_passes':int(result['recurrent_passes']),
            'loop_token_passes':int(result['loop_token_passes']),
            'mean_exit_depth':float(result['mean_exit_depth']) if result.get('mean_exit_depth') is not None else None,
            'wall_seconds':float(time.perf_counter()-started),'internal_wall_seconds':float(result.get('wall_seconds',0)),
            'stop_reason':str(result.get('stop_reason','unknown')),
            'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30)}
smoke=invoke(encode_prompt(CAL_BANK[0]),12,GEN_MODE,BASE_SEED)
assert smoke['outer_steps']>0 and smoke['loop_token_passes']>0
print('REAL CUDA smoke PASSED', {k:smoke[k] for k in ('outer_steps','loop_token_passes','wall_seconds','peak_vram_gb')})

### B5. Model-native semantic-failure discovery (correct boxed answers excluded)

In [ ]:
def natural_id(p):return p['problem_id']+'|natural|'+str(BASE_SEED)
def natural_run(p):
    tid=natural_id(p);old=get_rec('natural',tid)
    if old:return old
    r=invoke(encode_prompt(p),MAX_NEW,GEN_MODE,BASE_SEED)
    txt=decode_ids(r['token_ids']);v=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':p['problem_id'],'split':p['split'],
         'kind':p['kind'],'problem_fingerprint':p['problem_fingerprint'],'original_output':txt,
         'original_token_ids':r.pop('token_ids'),'evaluation':v,**r}
    save_rec('natural',obj)
    return obj

def cohort_from_natural(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    bykind=defaultdict(list)
    for p in bank:
        rec=get_rec('natural',natural_id(p))
        if rec and rec['evaluation']['eligible_semantic_failure']:bykind[p['kind']].append(p['problem_id'])
    if not all(len(bykind[k])>=targets[k] for k in targets):
        print('Eligible failures:',{k:len(bykind[k]) for k in targets},'required:',targets)
        return None
    subset=[x for k in targets for x in bykind[k][:targets[k]]]
    return subset

def natural_batch(split,start,limit=8):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    if cohort_from_natural(split):print('Already enough failures; skipping further candidate generation.');return
    for p in bank[start:start+limit]:
        if cohort_from_natural(split):break
        r=natural_run(p)
        print(p['problem_id'],r['evaluation']['success'],round(r['wall_seconds'],2),'s')
    print('Progress',len(all_recs('natural')),'native generations total.')

def freeze_cohorts():
    cp=cohort_from_natural('calibration');hp=cohort_from_natural('heldout')
    assert cp is not None and hp is not None,'Not enough model-native failures yet: continue natural-generation batches.'
    assert set(cp).isdisjoint(hp)
    content={'protocol_hash':RUN_SIG,'calibration':cp,'heldout':hp,'problem_bank_sha256':BANK_HASH}
    file=ROOT/'frozen_cohorts.json'
    if file.exists():assert json.loads(file.read_text())==content,'Frozen cohort differs; refusing test-set mutation.'
    else:atomic_write(file,content)
    print('FROZEN',len(cp),'calibration and',len(hp),'heldout native failures.')
    return content

### B6. Calibration candidate generation, 4 per short resumable cell
**Stop once each task quota is met.** These cells automatically skip already-selected splits. Do not choose a cohort based on held-out repair outcomes.

In [ ]:
natural_batch('calibration',0,4)

In [ ]:
natural_batch('calibration',4,4)

In [ ]:
natural_batch('calibration',8,4)

In [ ]:
natural_batch('calibration',12,4)

In [ ]:
natural_batch('calibration',16,4)

In [ ]:
natural_batch('calibration',20,4)

In [ ]:
natural_batch('calibration',24,4)

In [ ]:
natural_batch('calibration',28,4)

In [ ]:
natural_batch('calibration',32,4)

In [ ]:
natural_batch('calibration',36,4)

In [ ]:
natural_batch('calibration',40,4)

In [ ]:
natural_batch('calibration',44,4)

In [ ]:
natural_batch('calibration',48,4)

In [ ]:
natural_batch('calibration',52,4)

In [ ]:
natural_batch('calibration',56,4)

In [ ]:
natural_batch('calibration',60,4)

In [ ]:
natural_batch('calibration',64,4)

In [ ]:
natural_batch('calibration',68,4)

In [ ]:
natural_batch('calibration',72,4)

In [ ]:
natural_batch('calibration',76,4)

In [ ]:
natural_batch('calibration',80,4)

In [ ]:
natural_batch('calibration',84,4)

In [ ]:
natural_batch('calibration',88,4)

In [ ]:
natural_batch('calibration',92,4)

In [ ]:
natural_batch('calibration',96,4)

In [ ]:
natural_batch('calibration',100,4)

In [ ]:
natural_batch('calibration',104,4)

In [ ]:
natural_batch('calibration',108,4)

In [ ]:
natural_batch('calibration',112,4)

In [ ]:
natural_batch('calibration',116,4)

In [ ]:
natural_batch('calibration',120,4)

In [ ]:
natural_batch('calibration',124,4)

In [ ]:
natural_batch('calibration',128,4)

In [ ]:
natural_batch('calibration',132,4)

In [ ]:
natural_batch('calibration',136,4)

In [ ]:
natural_batch('calibration',140,4)

In [ ]:
natural_batch('calibration',144,4)

In [ ]:
natural_batch('calibration',148,4)

In [ ]:
natural_batch('calibration',152,4)

In [ ]:
natural_batch('calibration',156,4)

In [ ]:
natural_batch('calibration',160,4)

In [ ]:
natural_batch('calibration',164,4)

In [ ]:
natural_batch('calibration',168,4)

In [ ]:
natural_batch('calibration',172,4)

In [ ]:
natural_batch('calibration',176,4)

In [ ]:
natural_batch('calibration',180,4)

In [ ]:
natural_batch('calibration',184,4)

In [ ]:
natural_batch('calibration',188,4)

In [ ]:
natural_batch('calibration',192,4)

In [ ]:
natural_batch('calibration',196,4)

In [ ]:
natural_batch('calibration',200,4)

In [ ]:
natural_batch('calibration',204,4)

In [ ]:
natural_batch('calibration',208,4)

In [ ]:
natural_batch('calibration',212,4)

In [ ]:
natural_batch('calibration',216,4)

In [ ]:
natural_batch('calibration',220,4)

In [ ]:
natural_batch('calibration',224,4)

In [ ]:
natural_batch('calibration',228,4)

In [ ]:
natural_batch('calibration',232,4)

In [ ]:
natural_batch('calibration',236,4)

In [ ]:
natural_batch('calibration',240,4)

In [ ]:
natural_batch('calibration',244,4)

In [ ]:
natural_batch('calibration',248,4)

In [ ]:
natural_batch('calibration',252,4)

In [ ]:
natural_batch('calibration',256,4)

In [ ]:
natural_batch('calibration',260,4)

In [ ]:
print('Semantic failure availability (calibration):', cohort_from_natural('calibration'));print('Saved native records:',len(all_recs('natural')))

### B6. Heldout candidate generation, 4 per short resumable cell
**Stop once each task quota is met.** These cells automatically skip already-selected splits. Do not choose a cohort based on held-out repair outcomes.

In [ ]:
natural_batch('heldout',0,4)

In [ ]:
natural_batch('heldout',4,4)

In [ ]:
natural_batch('heldout',8,4)

In [ ]:
natural_batch('heldout',12,4)

In [ ]:
natural_batch('heldout',16,4)

In [ ]:
natural_batch('heldout',20,4)

In [ ]:
natural_batch('heldout',24,4)

In [ ]:
natural_batch('heldout',28,4)

In [ ]:
natural_batch('heldout',32,4)

In [ ]:
natural_batch('heldout',36,4)

In [ ]:
natural_batch('heldout',40,4)

In [ ]:
natural_batch('heldout',44,4)

In [ ]:
natural_batch('heldout',48,4)

In [ ]:
natural_batch('heldout',52,4)

In [ ]:
natural_batch('heldout',56,4)

In [ ]:
natural_batch('heldout',60,4)

In [ ]:
natural_batch('heldout',64,4)

In [ ]:
natural_batch('heldout',68,4)

In [ ]:
natural_batch('heldout',72,4)

In [ ]:
natural_batch('heldout',76,4)

In [ ]:
natural_batch('heldout',80,4)

In [ ]:
natural_batch('heldout',84,4)

In [ ]:
natural_batch('heldout',88,4)

In [ ]:
natural_batch('heldout',92,4)

In [ ]:
natural_batch('heldout',96,4)

In [ ]:
natural_batch('heldout',100,4)

In [ ]:
natural_batch('heldout',104,4)

In [ ]:
natural_batch('heldout',108,4)

In [ ]:
natural_batch('heldout',112,4)

In [ ]:
natural_batch('heldout',116,4)

In [ ]:
natural_batch('heldout',120,4)

In [ ]:
natural_batch('heldout',124,4)

In [ ]:
natural_batch('heldout',128,4)

In [ ]:
natural_batch('heldout',132,4)

In [ ]:
natural_batch('heldout',136,4)

In [ ]:
natural_batch('heldout',140,4)

In [ ]:
natural_batch('heldout',144,4)

In [ ]:
natural_batch('heldout',148,4)

In [ ]:
natural_batch('heldout',152,4)

In [ ]:
natural_batch('heldout',156,4)

In [ ]:
natural_batch('heldout',160,4)

In [ ]:
natural_batch('heldout',164,4)

In [ ]:
natural_batch('heldout',168,4)

In [ ]:
natural_batch('heldout',172,4)

In [ ]:
natural_batch('heldout',176,4)

In [ ]:
natural_batch('heldout',180,4)

In [ ]:
natural_batch('heldout',184,4)

In [ ]:
natural_batch('heldout',188,4)

In [ ]:
natural_batch('heldout',192,4)

In [ ]:
natural_batch('heldout',196,4)

In [ ]:
natural_batch('heldout',200,4)

In [ ]:
natural_batch('heldout',204,4)

In [ ]:
natural_batch('heldout',208,4)

In [ ]:
natural_batch('heldout',212,4)

In [ ]:
natural_batch('heldout',216,4)

In [ ]:
natural_batch('heldout',220,4)

In [ ]:
natural_batch('heldout',224,4)

In [ ]:
natural_batch('heldout',228,4)

In [ ]:
natural_batch('heldout',232,4)

In [ ]:
natural_batch('heldout',236,4)

In [ ]:
natural_batch('heldout',240,4)

In [ ]:
natural_batch('heldout',244,4)

In [ ]:
natural_batch('heldout',248,4)

In [ ]:
natural_batch('heldout',252,4)

In [ ]:
natural_batch('heldout',256,4)

In [ ]:
natural_batch('heldout',260,4)

In [ ]:
natural_batch('heldout',264,4)

In [ ]:
natural_batch('heldout',268,4)

In [ ]:
natural_batch('heldout',272,4)

In [ ]:
natural_batch('heldout',276,4)

In [ ]:
natural_batch('heldout',280,4)

In [ ]:
natural_batch('heldout',284,4)

In [ ]:
natural_batch('heldout',288,4)

In [ ]:
natural_batch('heldout',292,4)

In [ ]:
natural_batch('heldout',296,4)

In [ ]:
natural_batch('heldout',300,4)

In [ ]:
natural_batch('heldout',304,4)

In [ ]:
natural_batch('heldout',308,4)

In [ ]:
natural_batch('heldout',312,4)

In [ ]:
natural_batch('heldout',316,4)

In [ ]:
natural_batch('heldout',320,4)

In [ ]:
natural_batch('heldout',324,4)

In [ ]:
natural_batch('heldout',328,4)

In [ ]:
natural_batch('heldout',332,4)

In [ ]:
natural_batch('heldout',336,4)

In [ ]:
natural_batch('heldout',340,4)

In [ ]:
natural_batch('heldout',344,4)

In [ ]:
natural_batch('heldout',348,4)

In [ ]:
natural_batch('heldout',352,4)

In [ ]:
natural_batch('heldout',356,4)

In [ ]:
natural_batch('heldout',360,4)

In [ ]:
natural_batch('heldout',364,4)

In [ ]:
natural_batch('heldout',368,4)

In [ ]:
natural_batch('heldout',372,4)

In [ ]:
natural_batch('heldout',376,4)

In [ ]:
natural_batch('heldout',380,4)

In [ ]:
print('Semantic failure availability (heldout):', cohort_from_natural('heldout'));print('Saved native records:',len(all_recs('natural')))

### B7. Freeze semantic cohorts; this must pass before any repair evaluation

In [ ]:
COHORTS=freeze_cohorts()
PROBLEMS={p['problem_id']:p for p in CAL_BANK+TEST_BANK}
def failure(pid):
    p=PROBLEMS[pid];r=get_rec('natural',natural_id(p))
    assert r and r['evaluation']['eligible_semantic_failure'] and r['problem_fingerprint']==p['problem_fingerprint']
    return p,r

### B8. Prefix-preserving ALoDLM real repair calls; stop-on-error checkpoints

In [ ]:
def scope_cut(n,action):
    if action=='global':return 0
    if action not in ACTIONS:raise KeyError(action)
    frac={'tail25':.75,'tail50':.50,'tail75':.25}[action]
    return max(1,min(n-1,int(math.floor(n*frac))))

def repair_one(pid,action,seed,mode=REPAIR_MODE,stage='calibration'):
    assert action in ACTIONS
    tid=f'{stage}|{pid}|{action}|{mode}|{seed}'
    old=get_rec(stage,tid)
    if old:return old
    p,source=failure(pid)
    ids=source['original_token_ids']; n=len(ids);assert n>=4
    cut=scope_cut(n,action);preserved=ids[:cut];budget=max(1,n-cut)
    assert budget<=MAX_NEW
    r=invoke(encode_prompt(p)+preserved,budget,mode,seed)
    new_full=preserved+r['token_ids']
    assert new_full[:cut]==preserved, 'Prefix-preservation invariant failed.'
    txt=decode_ids(new_full)
    eval_=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'stage':stage,'problem_id':pid,'split':p['split'],
        'kind':p['kind'],'action':action,'mode':mode,'seed':int(seed),'original_response_tokens':n,
        'preserved_tokens':cut,'requested_new_tokens':budget,'preserved_fraction':cut/n,
        'exact_prefix_preserved':True,'original_output':source['original_output'],'repaired_output':txt,
        'evaluation':eval_,'public_accept':public_accept(txt,p),
        'original_repeated_exactly':txt==source['original_output'],
        **{k:v for k,v in r.items() if k!='token_ids'}}
    save_rec(stage,obj)
    return obj

def run_grid_batch(stage,grid,start,count=8):
    for pid,action,mode,seed in grid[start:start+count]:
        r=repair_one(pid,action,seed,mode,stage)
        print(f'{stage} {r["problem_id"]} {action} {mode} {seed}: success={r["evaluation"]["success"]} LTP={r["loop_token_passes"]}')
    print('Saved',len(all_recs(stage)),'stage records')

CAL_GRID=[(pid,a,REPAIR_MODE,s) for pid in COHORTS['calibration'] for a in ACTIONS for s in CAL_SEEDS]
assert len(CAL_GRID)==96

### B9. Calibrate on the 12 calibration failures (96 repair proposals)

In [ ]:
run_grid_batch('calibration',CAL_GRID,0,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,4,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,8,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,12,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,16,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,20,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,24,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,28,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,32,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,36,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,40,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,44,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,48,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,52,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,56,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,60,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,64,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,68,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,72,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,76,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,80,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,84,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,88,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,92,4)

### B10. Fit and freeze a policy using calibration outcomes ONLY

In [ ]:
def calibrate():
    data=all_recs('calibration');exp={f'calibration|{pid}|{a}|{REPAIR_MODE}|{seed}' for pid in COHORTS['calibration'] for a in ACTIONS for seed in CAL_SEEDS}
    assert len(data)==96 and set(r['trial_id'] for r in data)==exp,'Calibration incomplete; DO NOT inspect heldout results or fit policy yet.'
    pools=defaultdict(list)
    for r in data:
        p,src=failure(r['problem_id'])
        f=features(src['original_output'],p,len(src['original_token_ids']))
        pools[(feat_bucket(f),r['action'])].append(r)
        pools[(p['kind'],r['action'])].append(r)
        pools[('all',r['action'])].append(r)
    # Predeclared Laplace prior and lightweight cost penalty. No target-label features at deployment.
    def value(group,action):
        rows=pools.get((group,action),[])
        if not rows:return None
        succ=sum(r['evaluation']['success'] for r in rows)
        ltp=statistics.mean(r['loop_token_passes'] for r in rows)
        glob=pools[('all','global')]
        relative_cost=ltp/max(1,statistics.mean(r['loop_token_passes'] for r in glob))
        return (succ+1)/(len(rows)+2)-0.05*relative_cost
    fallback=max(ACTIONS,key=lambda a:(value('all',a),-ACTIONS.index(a)))
    kinds=sorted({p['kind'] for p in CAL_BANK})
    mapping={}
    for kind in kinds:
        for has_final in (True,False):
            buck=kind+'|'+('final' if has_final else 'no_final')
            group=buck if min(len(pools[(buck,a)]) for a in ACTIONS)>=6 else kind
            mapping[buck]=max(ACTIONS,key=lambda a:(value(group,a),-ACTIONS.index(a)))
    calib_summary={a:{'trials':len(pools[('all',a)]),'successes':sum(r['evaluation']['success'] for r in pools[('all',a)]),
                      'mean_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in pools[('all',a)])} for a in ACTIONS}
    frozen={'protocol_hash':RUN_SIG,'calibration_trial_hash':sha(sorted((r['trial_id'],r['evaluation']['success']) for r in data)),
            'best_fixed':fallback,'mapping':mapping,'lambda_cost':0.05,'action_summary':calib_summary,'trained_only_on':'calibration'}
    path=ROOT/'FROZEN_POLICY.json'
    if path.exists():assert json.loads(path.read_text())==frozen,'Cannot refit a different policy after freezing.'
    else:atomic_write(path,frozen)
    return frozen
POLICY=calibrate()
print('FROZEN POLICY:',json.dumps(POLICY,indent=2))

### B11. Held-out policy evaluations: 24 semantic failures × 5 policies × 3 seeds = 360 logical outcomes
Five policies share generated candidates when action+seed match. All unique GPU primitives are logged separately.

In [ ]:
def choose_action(policy,pid,seed):
    p,src=failure(pid)
    if policy=='global':return 'global'
    if policy=='tail50':return 'tail50'
    if policy=='best_fixed':return POLICY['best_fixed']
    if policy=='adaptive':
        f=features(src['original_output'],p,len(src['original_token_ids']))
        return POLICY['mapping'][feat_bucket(f)]
    if policy=='random_scope':
        # Precommitted randomization independent of targets/outcomes; includes global.
        seed_int=int(sha({'pid':pid,'seed':seed,'policy':'random_scope'})[:12],16)
        return random.Random(seed_int).choice(ACTIONS)
    raise KeyError(policy)
def heldout_one(pid,policy,seed):
    action=choose_action(policy,pid,seed)
    primitive=repair_one(pid,action,seed,REPAIR_MODE,stage='test_primitives')
    tid=f'heldout|{pid}|{policy}|{seed}'
    old=get_rec('heldout',tid)
    if old:return old
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'policy':policy,'action':action,'seed':seed,
       'primitive_id':primitive['trial_id'],'evaluation':primitive['evaluation'],'loop_token_passes':primitive['loop_token_passes'],
       'recurrent_passes':primitive['recurrent_passes'],'wall_seconds':primitive['wall_seconds'],
       'preserved_fraction':primitive['preserved_fraction'],'public_accept':primitive['public_accept'],
       'original_repeated_exactly':primitive['original_repeated_exactly']}
    save_rec('heldout',r)
    return r
TEST_GRID=[(pid,pol,s) for pid in COHORTS['heldout'] for pol in POLICIES for s in TEST_SEEDS]
assert len(TEST_GRID)==360
# Policies may share the same physical candidate (e.g. adaptive=global). This is logged as
# a shared primitive, never represented as two independent GPU samples or charged twice in resources.
def heldout_batch(start,count=8):
    for pid,pol,s in TEST_GRID[start:start+count]:
        r=heldout_one(pid,pol,s)
        print(pid,pol,r['action'],s,'SUCCESS',r['evaluation']['success'],'LTP',r['loop_token_passes'])
    print('Heldout logical outcomes',len(all_recs('heldout')),'/',len(TEST_GRID),
          '| unique physical GPU candidates',len(all_recs('test_primitives')))

In [ ]:
heldout_batch(0,5)

In [ ]:
heldout_batch(5,5)

In [ ]:
heldout_batch(10,5)

In [ ]:
heldout_batch(15,5)

In [ ]:
heldout_batch(20,5)

In [ ]:
heldout_batch(25,5)

In [ ]:
heldout_batch(30,5)

In [ ]:
heldout_batch(35,5)

In [ ]:
heldout_batch(40,5)

In [ ]:
heldout_batch(45,5)

In [ ]:
heldout_batch(50,5)

In [ ]:
heldout_batch(55,5)

In [ ]:
heldout_batch(60,5)

In [ ]:
heldout_batch(65,5)

In [ ]:
heldout_batch(70,5)

In [ ]:
heldout_batch(75,5)

In [ ]:
heldout_batch(80,5)

In [ ]:
heldout_batch(85,5)

In [ ]:
heldout_batch(90,5)

In [ ]:
heldout_batch(95,5)

In [ ]:
heldout_batch(100,5)

In [ ]:
heldout_batch(105,5)

In [ ]:
heldout_batch(110,5)

In [ ]:
heldout_batch(115,5)

In [ ]:
heldout_batch(120,5)

In [ ]:
heldout_batch(125,5)

In [ ]:
heldout_batch(130,5)

In [ ]:
heldout_batch(135,5)

In [ ]:
heldout_batch(140,5)

In [ ]:
heldout_batch(145,5)

In [ ]:
heldout_batch(150,5)

In [ ]:
heldout_batch(155,5)

In [ ]:
heldout_batch(160,5)

In [ ]:
heldout_batch(165,5)

In [ ]:
heldout_batch(170,5)

In [ ]:
heldout_batch(175,5)

In [ ]:
heldout_batch(180,5)

In [ ]:
heldout_batch(185,5)

In [ ]:
heldout_batch(190,5)

In [ ]:
heldout_batch(195,5)

In [ ]:
heldout_batch(200,5)

In [ ]:
heldout_batch(205,5)

In [ ]:
heldout_batch(210,5)

In [ ]:
heldout_batch(215,5)

In [ ]:
heldout_batch(220,5)

In [ ]:
heldout_batch(225,5)

In [ ]:
heldout_batch(230,5)

In [ ]:
heldout_batch(235,5)

In [ ]:
heldout_batch(240,5)

In [ ]:
heldout_batch(245,5)

In [ ]:
heldout_batch(250,5)

In [ ]:
heldout_batch(255,5)

In [ ]:
heldout_batch(260,5)

In [ ]:
heldout_batch(265,5)

In [ ]:
heldout_batch(270,5)

In [ ]:
heldout_batch(275,5)

In [ ]:
heldout_batch(280,5)

In [ ]:
heldout_batch(285,5)

In [ ]:
heldout_batch(290,5)

In [ ]:
heldout_batch(295,5)

In [ ]:
heldout_batch(300,5)

In [ ]:
heldout_batch(305,5)

In [ ]:
heldout_batch(310,5)

In [ ]:
heldout_batch(315,5)

In [ ]:
heldout_batch(320,5)

In [ ]:
heldout_batch(325,5)

In [ ]:
heldout_batch(330,5)

In [ ]:
heldout_batch(335,5)

In [ ]:
heldout_batch(340,5)

In [ ]:
heldout_batch(345,5)

In [ ]:
heldout_batch(350,5)

In [ ]:
heldout_batch(355,5)

### B12. Two-attempt budget study: adaptive→global vs global→global
24 held-out problems × 2 arms × 2 seeds = 96 logical evaluations. Actual GPU calls are separately counted.

In [ ]:
STAGE_ARMS=('adaptive_then_global','global_then_global')
STAGE_GRID=[(pid,arm,s) for pid in COHORTS['heldout'] for arm in STAGE_ARMS for s in STAGED_SEEDS]
assert len(STAGE_GRID)==96

def staged_one(pid,arm,seed):
    tid=f'staged|{pid}|{arm}|{seed}'
    old=get_rec('staged',tid)
    if old:return old
    p,src=failure(pid)
    actions=(choose_action('adaptive',pid,seed),'global') if arm=='adaptive_then_global' else ('global','global')
    calls=[];accepted=None
    for j,action in enumerate(actions):
        # Distinct seeds across two attempts; no test-time gold answers involved in stopping.
        attempt_seed=seed+j*777
        primitive=repair_one(pid,action,attempt_seed,REPAIR_MODE,stage='staged_primitives')
        calls.append(primitive)
        if primitive['public_accept']:
            accepted=primitive;break
    # If neither meets the public gate, the last candidate is the output (not oracle-selected).
    selected=accepted if accepted else calls[-1]
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'arm':arm,'seed':seed,
       'actions_used':[x['action'] for x in calls],'proposal_count':len(calls),
       'proposal_ids':[x['trial_id'] for x in calls],
       'public_accept':selected['public_accept'],'evaluation':selected['evaluation'],
       'loop_token_passes':sum(x['loop_token_passes'] for x in calls),
       'wall_seconds':sum(x['wall_seconds'] for x in calls),
       'recurrent_passes':sum(x['recurrent_passes'] for x in calls)}
    save_rec('staged',r)
    return r

def staged_batch(start,count=6):
    for pid,arm,s in STAGE_GRID[start:start+count]:
        r=staged_one(pid,arm,s)
        print(pid,arm,s,'success',r['evaluation']['success'],'proposals',r['proposal_count'],
              'loop_token_passes',r['loop_token_passes'])
    print('Staged',len(all_recs('staged')),'/',len(STAGE_GRID))

In [ ]:
staged_batch(0,4)

In [ ]:
staged_batch(4,4)

In [ ]:
staged_batch(8,4)

In [ ]:
staged_batch(12,4)

In [ ]:
staged_batch(16,4)

In [ ]:
staged_batch(20,4)

In [ ]:
staged_batch(24,4)

In [ ]:
staged_batch(28,4)

In [ ]:
staged_batch(32,4)

In [ ]:
staged_batch(36,4)

In [ ]:
staged_batch(40,4)

In [ ]:
staged_batch(44,4)

In [ ]:
staged_batch(48,4)

In [ ]:
staged_batch(52,4)

In [ ]:
staged_batch(56,4)

In [ ]:
staged_batch(60,4)

In [ ]:
staged_batch(64,4)

In [ ]:
staged_batch(68,4)

In [ ]:
staged_batch(72,4)

In [ ]:
staged_batch(76,4)

In [ ]:
staged_batch(80,4)

In [ ]:
staged_batch(84,4)

In [ ]:
staged_batch(88,4)

In [ ]:
staged_batch(92,4)

### B13. Final analysis: exact trials, problem-level sign tests, compute accounting, recurrence and format compliance
The main conclusion uses held-out *distinct problems*, not repeated trial counts as independent evidence. Exploratory multiplicity is declared; no selective claiming of wins.

In [ ]:

from scipy.stats import beta,binomtest
import itertools

def exact_binomial_ci(k,n,alpha=.05):
    return ((0.0 if k==0 else float(beta.ppf(alpha/2,k,n-k+1))),
            (1.0 if k==n else float(beta.ppf(1-alpha/2,k+1,n-k))))

def summarize(rows,field):
    out=[]
    for arm in sorted(set(r[field] for r in rows)):
        rr=[r for r in rows if r[field]==arm]
        success=[r for r in rr if r['evaluation']['success']]
        distinct={r['problem_id'] for r in success}
        nproblem=len({r['problem_id'] for r in rr});pn=len(distinct)
        lo,hi=exact_binomial_ci(pn,nproblem)
        out.append({field:arm,'n_trials':len(rr),'success_trials':len(success),
                   'distinct_problems':nproblem,'problems_recovered':pn,
                   'exact95_problem_low':lo,'exact95_problem_high':hi,
                   'strict_format_successes':sum(bool(r['evaluation']['strict_correct']) for r in rr),
                   'mean_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in rr),
                   'mean_wall_seconds':statistics.mean(r['wall_seconds'] for r in rr)})
    return out

def paired_problem_comparison(records,a,b,key='policy'):
    ids=sorted({r['problem_id'] for r in records})
    table=[]
    for pid in ids:
        aa=[r for r in records if r['problem_id']==pid and r[key]==a]
        bb=[r for r in records if r['problem_id']==pid and r[key]==b]
        assert aa and bb and {r['seed'] for r in aa}=={r['seed'] for r in bb}, 'Unpaired seeds or missing arm'
        table.append((pid,int(any(r['evaluation']['success'] for r in aa)),int(any(r['evaluation']['success'] for r in bb))))
    wins=sum(x>y for _,x,y in table);losses=sum(x<y for _,x,y in table)
    p=binomtest(wins,wins+losses,.5).pvalue if wins+losses else 1.0
    return {'A':a,'B':b,'n_independent_problems':len(table),'A_only_problems':wins,
            'B_only_problems':losses,'both_or_neither':len(table)-wins-losses,
            'difference_in_problem_coverage':(wins-losses)/len(table),'exact_two_sided_p':float(p)}

required={'calibration':96,'heldout':360,'staged':96}
counts={s:len(all_recs(s)) for s in required}
assert counts==required, 'Incomplete experiment; do not freeze partial journal tables: '+str(counts)
assert len(COHORTS['calibration'])==12 and len(COHORTS['heldout'])==24
assert len(set(COHORTS['calibration']))==12 and len(set(COHORTS['heldout']))==24
assert not (set(COHORTS['calibration'])&set(COHORTS['heldout']))
for pid in COHORTS['calibration']+COHORTS['heldout']:
    p,src=failure(pid)
    assert src['evaluation']['eligible_semantic_failure']
    assert src['evaluation']['failure_type']=='definite_wrong'
for stage in required:
    for r in all_recs(stage):assert r['protocol_hash']==RUN_SIG
for st in ('calibration','test_primitives','staged_primitives'):
    for r in all_recs(st):assert r['exact_prefix_preserved']
rows=all_recs('heldout'); two=all_recs('staged')
H=pd.DataFrame(summarize(rows,'policy'))
S=pd.DataFrame(summarize(two,'arm'))
CAL=pd.DataFrame(summarize(all_recs('calibration'),'action'))
PAIR=pd.DataFrame([paired_problem_comparison(rows,'adaptive',other) for other in ('global','best_fixed','random_scope','tail50')]
                  +[paired_problem_comparison(two,'adaptive_then_global','global_then_global',key='arm')])
ART.mkdir(parents=True,exist_ok=True)
for name,frame in [('primary_heldout_policy_summary',H),('two_stage_summary',S),('calibration_summary',CAL),('paired_problem_tests',PAIR)]:
    frame.to_csv(ART/(name+'.csv'),index=False)

# Distinct physical calls counted once, even where policies share a primitive.
compute={'physical_single_trial_proposals':len(all_recs('test_primitives')),
         'logical_heldout_policy_evaluations':len(rows),
         'physical_two_stage_proposals':len(all_recs('staged_primitives')),
         'logical_two_stage_evaluations':len(two),
         'original_semantic_failures':{'calibration':12,'heldout':24},
         'base_semantic_failure_counts':Counter(r['kind'] for r in all_recs('natural') if r['evaluation']['eligible_semantic_failure']),
         'base_format_only_or_unanswered_counts':Counter(r['evaluation']['failure_type'] for r in all_recs('natural'))}
atomic_write(ART/'compute_and_eligibility_audit.json',compute)

fig,axes=plt.subplots(figsize=(9,4.4))
Hsort=H.sort_values('policy')
axes.bar(Hsort['policy'],Hsort['problems_recovered'])
axes.set_ylabel('Distinct held-out semantic errors recovered (out of 24)')
axes.set_title('Repair policy coverage; repeated seeds clustered by problem')
plt.xticks(rotation=18,ha='right');plt.tight_layout()
fig.savefig(ART/'figure_semantic_problem_recovery.png',dpi=175)
fig.savefig(ART/'figure_semantic_problem_recovery.pdf');plt.close(fig)
fig,ax=plt.subplots(figsize=(8.4,4.5))
for _,r in H.iterrows():
    ax.scatter(r['mean_loop_token_passes'],r['problems_recovered'],s=70)
    ax.annotate(r['policy'],(r['mean_loop_token_passes'],r['problems_recovered']),xytext=(4,5),textcoords='offset points')
ax.set_xlabel('Mean native loop-token passes / policy trial')
ax.set_ylabel('Distinct held-out problems recovered')
ax.set_title('Measured compute and recovery (not assumed compute-matched)');plt.tight_layout()
fig.savefig(ART/'figure_compute_recovery.png',dpi=175)
fig.savefig(ART/'figure_compute_recovery.pdf');plt.close(fig)

# Holm multiplicity procedure for four primary exploratory contrasts.
P=PAIR.iloc[:4].copy().sort_values('exact_two_sided_p').reset_index(drop=True)
P['holm_adjusted_p']=1.0
v=0.0
for i in range(len(P)):
    v=max(v,float(P.loc[i,'exact_two_sided_p'])*(len(P)-i))
    P.loc[i,'holm_adjusted_p']=min(1.0,v)
P.to_csv(ART/'primary_paired_with_holm.csv',index=False)

MAN={'protocol_hash':RUN_SIG,'fixed_bank_sha256':BANK_HASH,'frozen_policy_sha256':sha(POLICY),
     'status':'COMPLETE','stages':counts,'primary_metric':'distinct_semantically_wrong problems recovered',
     'limits':['Synthetic benchmark; independent external validation remains necessary',
               'Human audit of a subset recommended; parser may miss unconventional answer formats',
               'Model results may be negative; null results must be reported',
               'Trial evaluations reuse proposal outputs and are not independent model generations']}
atomic_write(ART/'research_manifest.json',MAN)
print(H[['policy','success_trials','problems_recovered','mean_loop_token_passes']].to_string(index=False))
print('\nPaired problem tests:\n',PAIR.to_string(index=False))
print('NB29 final analysis COMPLETE. Figures and tables in',ART)


### B14. Export current state (SAFE to run at ANY point, including a partial run)
If the semantic-failure quota is not met, export the partial archive and resume next session. **Partial results are never labelled as complete.**

In [ ]:

def export_nb29():
    dst=Path('/content/RepairScopeX_NB29_Semantic_Repair_results.zip')
    optional=[ROOT/'protocol.json',ROOT/'frozen_problem_bank.json',ROOT/'frozen_cohorts.json',ROOT/'FROZEN_POLICY.json']
    items=[]
    for p in optional:
        if p.is_file():items.append((p,'protocol/'+p.name))
    if ART.exists():
        for p in ART.rglob('*'):
            if p.is_file():items.append((p,'artifacts/'+str(p.relative_to(ART))))
    for p in CACHE.rglob('*.json'):
        items.append((p,'records/'+str(p.relative_to(CACHE))))
    # An NB21 ZIP is included unmodified, as a SEPARATE experimental phase if previously finished.
    prev=Path('/content/drive/MyDrive/RepairScopeX/RSX29_Combined_Outputs/RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip')
    if prev.is_file():items.append((prev,'NB21_Separate_Completed_Results.zip'))
    assert len(set(name for _,name in items))==len(items)
    seen={name:hashlib.sha256(p.read_bytes()).hexdigest() for p,name in items}
    stages={s:len(all_recs(s)) for s in ('natural','calibration','heldout','test_primitives','staged','staged_primitives')}
    needed={'calibration':96,'heldout':360,'staged':96}
    is_complete=(all(stages[s]==v for s,v in needed.items()) and
                 (ART/'research_manifest.json').exists())
    details={'protocol_hash':RUN_SIG,'semantic_eligibility':'explicitly identifiable WRONG answers only',
             'partial':not is_complete,'stage_counts':stages,'required':needed,
             'nb21_included':prev.is_file(),'sha256':seen}
    with zipfile.ZipFile(dst,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        z.writestr('EXPORT_STATUS.json',json.dumps(details,indent=2,default=str))
        for p,name in sorted(items,key=lambda t:t[1]):z.write(p,name)
    with zipfile.ZipFile(dst) as z:
        assert z.testzip() is None
        for name,digest in seen.items():assert hashlib.sha256(z.read(name)).hexdigest()==digest
    print('Export',dst,'partial?',not is_complete,'saved stages',stages,'NB21 included',prev.is_file())
    return dst
NB29_ZIP=export_nb29()
from google.colab import files
files.download(str(NB29_ZIP))


### Final research interpretation and sharing

Return the combined ZIP (`RepairScopeX_NB29_Semantic_Repair_results.zip`), plus the MDLM NB21 ZIP if you exported that separately. 

**Strict separation:** NB21's 64 controlled MDLM infills and NB29's ALoDLM semantic-failure trials belong to different architectures, tasks, precisions, and protocols. Never pool trial-level success rates as if exchangeable.

A positive adaptive result is publishable only after verification of semantic correctness, independent problem-level paired comparison, and honest inference-compute accounting. A null outcome is also a valid result and should not be hidden. External held-out data and additional model transfer are not included in the synthetic NB29 phase; they remain independent confirmation work.
